In [ ]:
import numpy as np
import math
from pprint import pprint
from collections import Counter, defaultdict
from itertools import combinations
from collections import deque
from itertools import permutations, product
from tqdm import tqdm


from helpers import board_check, combo_counter, all_symmetries, min_rotation, find_third, dedup, build_sym_table

In [2]:
sym_table = build_sym_table(n=3, d=3) #precomputing sym table is MUCH faster
all_count = defaultdict(set)

all_cards = []
for one in range(3):
    for two in range(3):
        for three in range(3):
            all_cards.append((one, two, three))

board_init_pairs = set(map(lambda x: min_rotation(x, sym_table), combinations(all_cards, 2)))

for start_one, start_two in tqdm(board_init_pairs):
    start = {start_one, start_two}
    seen = {min_rotation(start, sym_table)}
    q = deque([start])
    ends = []

    longest = 2

    while True:
        if len(q) == 0:
            break
        
        trial_board = q.popleft()

        if len(trial_board) > longest:
            longest = len(trial_board)
            # print(longest, len(q))

        cant = [find_third(a, b) for a, b in combinations(trial_board, 2)]
        nexts = [card for card in all_cards if card not in trial_board and card not in cant]
        if not nexts:
            ends.append(trial_board)
        else:
            for n in nexts: 
                canon = min_rotation(trial_board | {n}, sym_table)
                if canon not in seen:
                    q.append(set(canon))
                    seen.add(canon)

    # print(f'ends by len: {Counter([len(x) for x in ends])}')
    for end in ends:
        all_count[len(end)].add(frozenset(end))
all_count = {k: dedup(v, sym_table) for k, v in all_count.items()}
# pprint(all_count)



For n=3, d=3: 48 symmetries


100%|██████████| 20/20 [00:31<00:00,  1.58s/it]


In [3]:
largest_board = max(all_count.keys())
for board in all_count[largest_board]:
    all_good = True
    if board_check(list(board)) == (0, []):
        continue
    else:
        print('compute error')
        all_good = False
if all_good:
    print(f'For a 3d grid, the max number of cards has {largest_board} cards, where there are {len(all_count[largest_board])} unique boards.')


For a 3d grid, the max number of cards has 9 cards, where there are 62 unique boards.


# 4D version


In [7]:
sym_table = build_sym_table(n=3, d=4) #precomputing sym table is MUCH faster
all_count = defaultdict(set)

all_cards = []
for one in range(3):
    for two in range(3):
        for three in range(3):
            for four in range(3):
                all_cards.append((one, two, three, four))

board_init_pairs = set(map(lambda x: min_rotation(x, sym_table), combinations(all_cards, 2)))

for start_one, start_two in tqdm(board_init_pairs):
    start = {start_one, start_two}
    seen = {min_rotation(start, sym_table)}
    q = deque([start])
    ends = []

    longest = 2

    while True:
        if len(q) == 0:
            break
        
        trial_board = q.popleft()

        if len(trial_board) > longest:
            longest = len(trial_board)
            # print(longest, len(q))

        cant = [find_third(a, b) for a, b in combinations(trial_board, 2)]
        nexts = [card for card in all_cards if card not in trial_board and card not in cant]
        if not nexts:
            ends.append(trial_board)
        else:
            for n in nexts: 
                canon = min_rotation(trial_board | {n}, sym_table)
                if canon not in seen:
                    q.append(set(canon))
                    seen.add(canon)

    # print(f'ends by len: {Counter([len(x) for x in ends])}')
    for end in ends:
        all_count[len(end)].add(frozenset(end))
pprint(all_count)



  0%|          | 0/41 [01:20<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
largest_board = max(all_count.keys())
for board in all_count[largest_board]:
    all_good = True
    if board_check(list(board)) == (0, []):
        continue
    else:
        print('compute error')
        all_good = False
if all_good:
    print(f'For a 3d grid, the max number of cards has {largest_board} cards, where there are {len(all_count[largest_board])} unique boards.')


In [12]:
def all_symmetries_ref(coords, n=3):
    coords = list(coords); d = len(coords[0]); off = (n - 1) / 2
    def tr(p, perm, signs):
        c = [p[k] - off for k in range(d)]
        c = [signs[k] * c[perm[k]] for k in range(d)]
        return tuple(int(round(v + off)) for v in c)
    return {frozenset(tr(p, perm, signs) for p in coords)
            for perm in permutations(range(d))
            for signs in product((1, -1), repeat=d)}

for b in [{(0,0,0,0),(0,0,0,1)},
          {(0,0,0,0),(1,2,0,1),(2,1,1,0)},
          {(2,2,2,2),(0,1,2,0)}]:
    assert all_symmetries(b, sym_table) == all_symmetries_ref(b)

Now in C...

In [5]:
!gcc -O3 filter_mask.c -o filter_mask
!./filter_mask

Building symmetry table...
Built 384 symmetries
Canonical initial pairs: 41

Starting pair 1 / 41: {(0,0,0,1), (0,0,1,0)}
longest: 3, queue: 33, seen: 35
longest: 4, queue: 828, seen: 864
longest: 5, queue: 15498, seen: 16363
longest: 6, queue: 222115, seen: 238479
longest: 7, queue: 2398830, seen: 2637310
^C
